# Injection Demo — Plots for Update

Generates all key figures:
1. 2D light profiles (Plummer, King, EFF, Sersic)
2. PSF extraction from coadd
3. Raw profile vs PSF-convolved profile
4. Before / After / Difference panels
5. Postage stamp grid of injected clusters
6. Summary diagnostic panel

---
## 0. Setup

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle
from scipy.signal import fftconvolve
from scipy.ndimage import gaussian_filter

# Add pipeline to path
USERNAME = os.environ.get('USER', 'your_username')
PIPELINE_PATH = f'/home/{USERNAME}/INJECT/star-cluster-injection-pipeline'
if os.path.exists(PIPELINE_PATH):
    sys.path.insert(0, PIPELINE_PATH)
    print(f'Pipeline: {PIPELINE_PATH}')

from src.light_profiles import PlummerProfile, KingProfile, EFFProfile, SersicProfile, mag_to_flux

print('Imports OK')

---
## 1. Load Rubin Coadd + PSF

In [ ]:
from lsst.daf.butler import Butler
from lsst.geom import Point2D

DATA_ID = {'tract': 4431, 'patch': 17, 'band': 'i'}

butler = None
for repo, coll in [('dp02', '2.2i/runs/DP0.2'),
                    ('s3://butler-us-central1-dp02', '2.2i/runs/DP0.2'),
                    ('/repo/dc2', '2.2i/runs/DP0.2')]:
    try:
        print(f'Trying {repo!r}...')
        butler = Butler(repo, collections=coll)
        exposure = butler.get('deepCoadd', dataId=DATA_ID)
        print(f'  Connected!')
        break
    except Exception as e:
        print(f'  {type(e).__name__}: {str(e)[:80]}')
        butler = None

if butler is None:
    raise RuntimeError('Could not connect to Butler. Run on RSP.')

image = exposure.image.array.copy()
ZEROPOINT = 31.4  # Rubin nJy AB

# Extract PSF at center
cy, cx = image.shape[0]//2, image.shape[1]//2
psf_model = exposure.getPsf()
psf_arr = psf_model.computeImage(Point2D(float(cx), float(cy))).array.copy()
psf_arr = psf_arr / psf_arr.sum()
sh = psf_model.computeShape(Point2D(float(cx), float(cy)))
psf_fwhm = 2.355 * np.sqrt((sh.getIxx() + sh.getIyy()) / 2.0)

print(f'Image     : {image.shape}')
print(f'PSF FWHM  : {psf_fwhm:.2f} px = {psf_fwhm*0.2:.2f} arcsec')
print(f'PSF kernel: {psf_arr.shape}')

---
## 2. Plot: 2D Light Profiles

Show all four supported profile types side by side.

In [ ]:
flux = mag_to_flux(21.0, zero_point=ZEROPOINT)
r_half = 10.0  # pixels
stamp_size = 101
half = stamp_size // 2
yy, xx = np.mgrid[:stamp_size, :stamp_size]
dx, dy = xx - half, yy - half

profiles = {
    'Plummer':  PlummerProfile(total_flux=flux, r_half=r_half),
    'King':     KingProfile(total_flux=flux, r_half=r_half, concentration=30),
    'EFF':      EFFProfile(total_flux=flux, r_half=r_half, gamma=2.5),
    'Sérsic':   SersicProfile(total_flux=flux, r_half=r_half, sersic_n=2.0),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, (name, prof) in enumerate(profiles.items()):
    stamp = prof.surface_brightness(dx, dy)

    # Top row: 2D image (log scale)
    ax = axes[0, i]
    stamp_pos = np.clip(stamp, 1e-12, None)
    im = ax.imshow(stamp_pos, cmap='inferno', origin='lower',
                   norm=LogNorm(vmin=stamp_pos.max()*1e-4, vmax=stamp_pos.max()))
    circ = Circle((half, half), r_half, fill=False, ec='cyan', lw=1.5, ls='--')
    ax.add_patch(circ)
    ax.set_title(f'{name}\nr_half={r_half} px', fontsize=11)
    if i == 0:
        ax.set_ylabel('Y (pixels)')
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Bottom row: radial profile
    ax = axes[1, i]
    rr = np.sqrt(dx**2 + dy**2).ravel()
    vals = stamp.ravel()
    order = np.argsort(rr)
    ax.semilogy(rr[order], vals[order], '.', ms=0.5, alpha=0.3, color='navy')
    ax.axvline(r_half, color='red', ls='--', lw=1.5, label=f'r_half={r_half}')
    ax.set_xlabel('Radius (px)')
    if i == 0:
        ax.set_ylabel('Surface brightness')
    ax.set_title(f'Radial profile')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xlim(0, half)

fig.suptitle('Supported 2D Light Profiles (mag=21, r_half=10 px)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_light_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_light_profiles.png')

---
## 3. Plot: Raw Profile vs PSF-Convolved

Show what the PSF does to an injected cluster.

In [ ]:
test_mags = [20.0, 22.0, 24.0]
test_r_halfs = [5.0, 10.0, 20.0]

fig, axes = plt.subplots(3, 3, figsize=(14, 14))

for row, mag in enumerate(test_mags):
    for col, rh in enumerate(test_r_halfs):
        f = mag_to_flux(mag, zero_point=ZEROPOINT)
        prof = PlummerProfile(total_flux=f, r_half=rh)
        sz = max(61, int(rh * 10) | 1)
        h = sz // 2
        yg, xg = np.mgrid[:sz, :sz]
        raw = prof.surface_brightness(xg - h, yg - h)
        conv = fftconvolve(raw, psf_arr, mode='same')

        ax = axes[row, col]
        # Show convolved on left half, raw on right half
        combined = np.zeros_like(raw)
        combined[:, :h] = conv[:, :h]
        combined[:, h:] = raw[:, h:]
        vmax = max(raw.max(), conv.max())
        ax.imshow(combined, cmap='inferno', origin='lower', vmin=0, vmax=vmax * 0.3)
        ax.axvline(h, color='white', lw=0.8, ls='-', alpha=0.7)
        ax.text(h * 0.3, sz - 5, 'PSF-conv', color='white', fontsize=8, ha='center')
        ax.text(h * 1.7, sz - 5, 'Raw', color='white', fontsize=8, ha='center')
        ax.set_title(f'mag={mag}, r½={rh} px', fontsize=10)
        if col == 0:
            ax.set_ylabel(f'mag={mag}')
        ax.tick_params(labelsize=6)

fig.suptitle(f'Raw vs PSF-convolved Plummer profiles (PSF FWHM={psf_fwhm:.1f} px)', fontsize=13)
plt.tight_layout()
plt.savefig('fig_psf_convolution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_psf_convolution.png')

---
## 4. Inject Clusters into the Coadd

In [ ]:
np.random.seed(42)

N_CLUSTERS = 30
MAG_MIN, MAG_MAX = 19.0, 25.0
R_HALF_MIN, R_HALF_MAX = 3.0, 25.0
EDGE_BUFFER = 200

ny, nx = image.shape
injected_image = image.copy().astype(np.float64)
injection_info = []

for i in range(N_CLUSTERS):
    # Random position, magnitude, size
    x = np.random.randint(EDGE_BUFFER, nx - EDGE_BUFFER)
    y = np.random.randint(EDGE_BUFFER, ny - EDGE_BUFFER)
    mag = np.random.uniform(MAG_MIN, MAG_MAX)
    rh = 10 ** np.random.uniform(np.log10(R_HALF_MIN), np.log10(R_HALF_MAX))
    flux = mag_to_flux(mag, zero_point=ZEROPOINT)

    # Random profile type
    ptype = np.random.choice(['plummer', 'king', 'eff', 'sersic'])
    if ptype == 'plummer':
        prof = PlummerProfile(total_flux=flux, r_half=rh)
    elif ptype == 'king':
        prof = KingProfile(total_flux=flux, r_half=rh, concentration=30)
    elif ptype == 'eff':
        prof = EFFProfile(total_flux=flux, r_half=rh, gamma=2.5)
    else:
        prof = SersicProfile(total_flux=flux, r_half=rh, sersic_n=2.0)

    # Generate stamp
    sz = max(61, int(rh * 10) | 1)
    h = sz // 2
    yg, xg = np.mgrid[:sz, :sz]
    raw_stamp = prof.surface_brightness(xg - h, yg - h)

    # Get local PSF and convolve
    pos = Point2D(float(x), float(y))
    try:
        local_psf = psf_model.computeImage(pos).array.copy()
        local_psf = local_psf / local_psf.sum()
    except:
        local_psf = psf_arr
    conv_stamp = fftconvolve(raw_stamp, local_psf, mode='same')

    # Add Poisson noise
    pos_mask = conv_stamp > 0
    noisy = conv_stamp.copy()
    noisy[pos_mask] = np.random.poisson(conv_stamp[pos_mask])

    # Place into image
    y0 = max(0, y - h)
    y1 = min(ny, y - h + sz)
    x0 = max(0, x - h)
    x1 = min(nx, x - h + sz)
    sy0 = y0 - (y - h)
    sy1 = sy0 + (y1 - y0)
    sx0 = x0 - (x - h)
    sx1 = sx0 + (x1 - x0)
    injected_image[y0:y1, x0:x1] += noisy[sy0:sy1, sx0:sx1]

    injection_info.append({
        'id': i, 'x': x, 'y': y,
        'magnitude': mag, 'r_half': rh,
        'profile': ptype,
        'flux_injected': float(noisy.sum()),
        'peak': float(noisy.max()),
        'raw_stamp': raw_stamp,
        'conv_stamp': conv_stamp,
    })

injected_image = injected_image.astype(image.dtype)
print(f'Injected {N_CLUSTERS} clusters')
print(f'  Mag range : [{MAG_MIN}, {MAG_MAX}]')
print(f'  r_half    : [{R_HALF_MIN}, {R_HALF_MAX}] px')
print(f'  PSF FWHM  : {psf_fwhm:.2f} px')

---
## 5. Plot: Before / After / Difference (Full Image)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

vmin, vmax = np.percentile(image, [1, 99])

# Before
axes[0].imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title('Original Coadd (before injection)', fontsize=12)

# After
axes[1].imshow(injected_image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
for info in injection_info:
    c = plt.cm.plasma(np.clip((info['magnitude'] - MAG_MIN) / (MAG_MAX - MAG_MIN), 0, 1))
    circ = Circle((info['x'], info['y']), info['r_half'] * 2,
                  fill=False, ec=c, lw=1.2, alpha=0.8)
    axes[1].add_patch(circ)
axes[1].set_title(f'After injection ({N_CLUSTERS} clusters)', fontsize=12)

# Difference
diff = (injected_image.astype(float) - image.astype(float))
diff_show = diff.copy()
diff_show[diff_show <= 0] = np.nan
valid = diff_show[np.isfinite(diff_show)]
if len(valid) > 0:
    im = axes[2].imshow(diff_show, cmap='hot', origin='lower',
                         norm=LogNorm(vmin=np.percentile(valid, 5), vmax=np.nanmax(valid)))
    plt.colorbar(im, ax=axes[2], shrink=0.8, label='Injected flux (nJy)')
axes[2].set_title('Difference (injected signal only)', fontsize=12)

for ax in axes:
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')

fig.suptitle(f'Star Cluster Injection — {DATA_ID["band"]}-band, tract {DATA_ID["tract"]}, patch {DATA_ID["patch"]}',
             fontsize=14)
plt.tight_layout()
plt.savefig('fig_before_after_diff.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_before_after_diff.png')

---
## 6. Plot: Postage Stamps (Before / After / Cluster Only)

In [ ]:
# Pick 8 clusters spanning the magnitude range
sorted_info = sorted(injection_info, key=lambda d: d['magnitude'])
n_stamps = min(8, len(sorted_info))
pick_idx = np.linspace(0, len(sorted_info) - 1, n_stamps, dtype=int)
picked = [sorted_info[i] for i in pick_idx]

fig, axes = plt.subplots(3, n_stamps, figsize=(2.5 * n_stamps, 7.5))

for col, info in enumerate(picked):
    cx_i, cy_i = int(info['x']), int(info['y'])
    r = max(int(info['r_half'] * 4), 25)

    y0 = max(0, cy_i - r)
    y1 = min(ny, cy_i + r)
    x0 = max(0, cx_i - r)
    x1 = min(nx, cx_i + r)

    stamp_orig = image[y0:y1, x0:x1].astype(float)
    stamp_inj = injected_image[y0:y1, x0:x1].astype(float)
    stamp_diff = stamp_inj - stamp_orig

    v1, v2 = np.percentile(stamp_inj, [1, 99.5])

    # Row 0: before
    axes[0, col].imshow(stamp_orig, cmap='gray', origin='lower', vmin=v1, vmax=v2)
    axes[0, col].set_title(f'mag={info["magnitude"]:.1f}', fontsize=9)

    # Row 1: after
    axes[1, col].imshow(stamp_inj, cmap='gray', origin='lower', vmin=v1, vmax=v2)
    axes[1, col].set_title(f'r½={info["r_half"]:.1f} px', fontsize=9)

    # Row 2: cluster only
    d_pos = np.clip(stamp_diff, 0, None)
    if d_pos.max() > 0:
        axes[2, col].imshow(d_pos, cmap='inferno', origin='lower',
                            norm=LogNorm(vmin=max(d_pos[d_pos > 0].min(), 1e-2),
                                         vmax=d_pos.max()))
    axes[2, col].set_title(f'{info["profile"]}', fontsize=9)

    # Mark center
    for row in range(3):
        axes[row, col].plot(cx_i - x0, cy_i - y0, 'r+', ms=8, mew=1)
        axes[row, col].tick_params(labelsize=5)

axes[0, 0].set_ylabel('Before', fontsize=11)
axes[1, 0].set_ylabel('After', fontsize=11)
axes[2, 0].set_ylabel('Cluster only', fontsize=11)

fig.suptitle('Postage Stamps — sorted by magnitude (bright → faint)', fontsize=13)
plt.tight_layout()
plt.savefig('fig_postage_stamps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_postage_stamps.png')

---
## 7. Plot: Raw vs PSF-Convolved Stamps (from actual injection)

In [ ]:
n_show = min(6, len(injection_info))
pick = sorted(injection_info, key=lambda d: d['r_half'], reverse=True)[:n_show]

fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))

for col, info in enumerate(pick):
    raw = info['raw_stamp']
    conv = info['conv_stamp']
    vmax = max(raw.max(), conv.max()) * 0.5

    axes[0, col].imshow(raw, cmap='inferno', origin='lower', vmin=0, vmax=vmax)
    axes[0, col].set_title(f'{info["profile"]}\nmag={info["magnitude"]:.1f}, r½={info["r_half"]:.1f}', fontsize=8)

    axes[1, col].imshow(conv, cmap='inferno', origin='lower', vmin=0, vmax=vmax)
    axes[1, col].set_title(f'peak ratio: {conv.max()/raw.max():.2f}', fontsize=8)

    for row in range(2):
        axes[row, col].tick_params(labelsize=5)

axes[0, 0].set_ylabel('Raw analytical', fontsize=11)
axes[1, 0].set_ylabel('PSF-convolved', fontsize=11)

fig.suptitle(f'Effect of PSF Convolution (FWHM={psf_fwhm:.1f} px)', fontsize=13)
plt.tight_layout()
plt.savefig('fig_raw_vs_convolved.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_raw_vs_convolved.png')

---
## 8. Plot: Zoomed Regions

In [ ]:
# Pick 3 clusters at different magnitudes for zoomed comparison
bright = min(injection_info, key=lambda d: d['magnitude'])
mid = sorted(injection_info, key=lambda d: abs(d['magnitude'] - np.median([i['magnitude'] for i in injection_info])))[0]
faint = max(injection_info, key=lambda d: d['magnitude'])

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
labels = ['Brightest', 'Median', 'Faintest']

for row, (info, label) in enumerate(zip([bright, mid, faint], labels)):
    cx_i, cy_i = int(info['x']), int(info['y'])
    r = max(int(info['r_half'] * 5), 40)

    y0, y1 = max(0, cy_i - r), min(ny, cy_i + r)
    x0, x1 = max(0, cx_i - r), min(nx, cx_i + r)

    s_orig = image[y0:y1, x0:x1].astype(float)
    s_inj = injected_image[y0:y1, x0:x1].astype(float)
    s_diff = s_inj - s_orig

    v1, v2 = np.percentile(s_inj, [0.5, 99.5])

    axes[row, 0].imshow(s_orig, cmap='gray', origin='lower', vmin=v1, vmax=v2)
    axes[row, 0].set_ylabel(f'{label}\nmag={info["magnitude"]:.1f}\nr½={info["r_half"]:.1f}px\n{info["profile"]}',
                             fontsize=10)

    axes[row, 1].imshow(s_inj, cmap='gray', origin='lower', vmin=v1, vmax=v2)
    circ = Circle((cx_i - x0, cy_i - y0), info['r_half'],
                  fill=False, ec='cyan', lw=1.5, ls='--')
    axes[row, 1].add_patch(circ)

    d = np.clip(s_diff, 0, None)
    if d.max() > 0:
        axes[row, 2].imshow(d, cmap='inferno', origin='lower',
                            norm=LogNorm(vmin=max(1e-2, d[d > 0].min()), vmax=d.max()))

    for col in range(3):
        axes[row, col].plot(cx_i - x0, cy_i - y0, 'r+', ms=12, mew=1.5)

axes[0, 0].set_title('Before', fontsize=12)
axes[0, 1].set_title('After', fontsize=12)
axes[0, 2].set_title('Cluster Only', fontsize=12)

fig.suptitle('Zoomed Injection Examples — Bright / Median / Faint', fontsize=14)
plt.tight_layout()
plt.savefig('fig_zoomed_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_zoomed_examples.png')

---
## 9. Plot: Summary Diagnostic Panel

In [ ]:
mags = [i['magnitude'] for i in injection_info]
sizes = [i['r_half'] for i in injection_info]
fluxes = [i['flux_injected'] for i in injection_info]
peaks = [i['peak'] for i in injection_info]
bg_std = np.std(image)
snrs = [p / bg_std for p in peaks]
ptypes = [i['profile'] for i in injection_info]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# (0,0) Magnitude histogram
axes[0, 0].hist(mags, bins=12, edgecolor='black', color='steelblue', alpha=0.8)
axes[0, 0].axvline(np.median(mags), color='red', ls='--', lw=1.5, label=f'median={np.median(mags):.1f}')
axes[0, 0].set_xlabel('Magnitude')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Magnitude Distribution')
axes[0, 0].legend()

# (0,1) Size histogram
axes[0, 1].hist(sizes, bins=12, edgecolor='black', color='darkorange', alpha=0.8)
axes[0, 1].axvline(np.median(sizes), color='red', ls='--', lw=1.5, label=f'median={np.median(sizes):.1f}')
axes[0, 1].set_xlabel('r_half (pixels)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Size Distribution')
axes[0, 1].legend()

# (0,2) Profile type pie chart
from collections import Counter
counts = Counter(ptypes)
axes[0, 2].pie(counts.values(), labels=counts.keys(), autopct='%1.0f%%',
               colors=plt.cm.Set2.colors[:len(counts)])
axes[0, 2].set_title('Profile Types Used')

# (1,0) Flux vs magnitude
sc = axes[1, 0].scatter(mags, np.log10(np.array(fluxes) + 1), c=sizes, cmap='plasma', s=40, alpha=0.8)
plt.colorbar(sc, ax=axes[1, 0], label='r_half (px)')
axes[1, 0].set_xlabel('Magnitude')
axes[1, 0].set_ylabel('log10(Injected flux)')
axes[1, 0].set_title('Flux vs Magnitude')

# (1,1) Peak S/N vs magnitude
axes[1, 1].scatter(mags, snrs, c=sizes, cmap='plasma', s=40, alpha=0.8)
axes[1, 1].axhline(3, color='red', ls='--', label='S/N=3')
axes[1, 1].axhline(5, color='orange', ls='--', label='S/N=5')
axes[1, 1].set_xlabel('Magnitude')
axes[1, 1].set_ylabel('Peak S/N')
axes[1, 1].set_title('Peak S/N vs Magnitude')
axes[1, 1].set_yscale('log')
axes[1, 1].legend(fontsize=8)

# (1,2) Summary text
axes[1, 2].axis('off')
summary = (
    f'Injection Summary\n'
    f'─────────────────────\n'
    f'N clusters  : {N_CLUSTERS}\n'
    f'Band        : {DATA_ID["band"]}\n'
    f'Tract/Patch : {DATA_ID["tract"]}/{DATA_ID["patch"]}\n'
    f'Image       : {image.shape}\n'
    f'\n'
    f'Mag range   : [{min(mags):.1f}, {max(mags):.1f}]\n'
    f'r½ range    : [{min(sizes):.1f}, {max(sizes):.1f}] px\n'
    f'PSF FWHM    : {psf_fwhm:.2f} px ({psf_fwhm*0.2:.2f}\'\')\n'
    f'PSF kernel  : {psf_arr.shape}\n'
    f'Zeropoint   : {ZEROPOINT}\n'
    f'BG std      : {bg_std:.2f}\n'
    f'\n'
    f'Median S/N  : {np.median(snrs):.1f}\n'
    f'S/N > 3     : {sum(1 for s in snrs if s > 3)}/{N_CLUSTERS}\n'
    f'S/N > 5     : {sum(1 for s in snrs if s > 5)}/{N_CLUSTERS}\n'
    f'\n'
    f'Profiles    : {dict(counts)}'
)
axes[1, 2].text(0.1, 0.95, summary, transform=axes[1, 2].transAxes,
                fontsize=10, va='top', family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle('Injection Diagnostic Summary', fontsize=14)
plt.tight_layout()
plt.savefig('fig_summary_panel.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_summary_panel.png')

---
## 10. Plot: PSF Kernel + Spatial Variation

In [ ]:
# Sample PSF at 9 positions
margin = 300
xs = np.linspace(margin, nx - margin, 3)
ys = np.linspace(margin, ny - margin, 3)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fwhm_vals = []

for row, y_pos in enumerate(ys):
    for col, x_pos in enumerate(xs):
        pos = Point2D(float(x_pos), float(y_pos))
        try:
            k = psf_model.computeImage(pos).array.copy()
            s = psf_model.computeShape(pos)
            fw = 2.355 * np.sqrt((s.getIxx() + s.getIyy()) / 2.0)
            fwhm_vals.append(fw)
            axes[row, col].imshow(k, cmap='inferno', origin='lower')
            axes[row, col].set_title(f'({x_pos:.0f}, {y_pos:.0f})\nFWHM={fw:.2f} px', fontsize=9)
        except Exception as e:
            axes[row, col].text(0.5, 0.5, 'FAILED', ha='center', va='center',
                                transform=axes[row, col].transAxes, color='red')
        axes[row, col].tick_params(labelsize=5)

if fwhm_vals:
    fig.suptitle(f'PSF Spatial Variation — FWHM: [{min(fwhm_vals):.2f}, {max(fwhm_vals):.2f}] px\n'
                 f'Range: {max(fwhm_vals)-min(fwhm_vals):.3f} px = {(max(fwhm_vals)-min(fwhm_vals))*0.2:.3f} arcsec',
                 fontsize=12)
plt.tight_layout()
plt.savefig('fig_psf_spatial.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_psf_spatial.png')

---
## Files Generated

| File | Description |
|------|------------|
| `fig_light_profiles.png` | All 4 profile types: 2D + radial |
| `fig_psf_convolution.png` | Raw vs PSF-convolved grid |
| `fig_before_after_diff.png` | Full-image before/after/difference |
| `fig_postage_stamps.png` | 8 clusters: before/after/cluster-only stamps |
| `fig_raw_vs_convolved.png` | Actual injected stamps raw vs convolved |
| `fig_zoomed_examples.png` | 3 zoomed examples: bright/median/faint |
| `fig_summary_panel.png` | Diagnostic summary: histograms, S/N, stats |
| `fig_psf_spatial.png` | PSF kernel at 9 positions across coadd |

In [ ]:
import glob
figs = sorted(glob.glob('fig_*.png'))
print(f'Generated {len(figs)} figures:')
for f in figs:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f:40s} {size_kb:6.0f} KB')